In [ ]:
#!/usr/bin/env python3
"""
Multi-objective, multi-concept optimization with pydace + NSGA-II.

Main changes compared to the PySR + SCIP version:
A) Surrogates:
   - PySR regressors/classifier → pydace.Dace Kriging models
     (one Dace per f_i, g_j, and optional clf_success).

B) Optimizer:
   - SCIP MINLP (via Pyomo & symbolic expressions) → NSGA-II (pymoo)
     used for *constrained single-objective* surrogate optimization:
       1) Minimize f1 surrogate
       2) Minimize f2 surrogate
       3) Minimize Tchebycheff scalarization surrogate for multiple z_targets

This version is fully vectorized on the surrogate side:
 - DACE predictions are batched (X is (N,D)).
 - NSGA-II uses pymoo.core.problem.Problem with batched _evaluate.
"""

import os
import json
import math
import warnings
import random
from pathlib import Path
from functools import reduce
import operator

import numpy as np
import pandas as pd
from scipy.stats import qmc
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import pickle
from concurrent.futures import ProcessPoolExecutor
import copy
# ======================================================
# Import pydacefit
# ======================================================
try:
    from pydacefit.dace import DACE as Dace
    from pydacefit.regr import regr_linear, regr_quadratic
    from pydacefit.corr import corr_gauss, corr_exp, corr_spherical
except ImportError:
    raise ImportError("pydacefit is required. Install via: pip install pydacefit")

# New: pymoo (NSGA-II)
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize

# Import problems and evaluation functions
from Problems import (
    problem_dtlz2,
    problem_zdt3,
    problem_bnh,
    problem_tnk,
    problem_simulator,
    problem_welded_beam,
)

# ----------------------------------------------------
# Environment & warnings
# ----------------------------------------------------
os.environ["DYLD_LIBRARY_PATH"] = "/usr/local/Cellar/gcc/14.2.0_1/lib/gcc/14"
warnings.filterwarnings("ignore", category=UserWarning, module="pydace")

ROUND_DECIMALS = 12
IDEAL_NADIR_PREC = 1e-6   # precision for ideal/nadir & F-quantization


def round_array(arr, decimals=ROUND_DECIMALS):
    """Round numpy array or scalar to the given decimals."""
    return np.round(np.asarray(arr, dtype=float), decimals)


def round_to_precision(arr, prec=IDEAL_NADIR_PREC):
    """
    Quantize array to a given precision, e.g. 1e-6.
    Used for F-values when computing/global ideal & nadir, and for scaling.
    """
    arr = np.asarray(arr, dtype=float)
    return np.round(arr / prec) * prec


def to_2d_rounded(x, decimals=ROUND_DECIMALS):
    """Ensure x is 2D float array and rounded."""
    xv = np.atleast_2d(x).astype(float)
    return np.round(xv, decimals)


# ----------------------------------------------------
# LHS (QMC) sampler
# ----------------------------------------------------
def sample_lhs_qmc(n_samples, dim, lb, ub):
    sampler = qmc.LatinHypercube(d=dim)
    U = sampler.random(n=n_samples)
    return qmc.scale(U, lb, ub)


# ====================================================
# Dominance, feasibility & distance utilities
# ====================================================

def normalized_distance_to_archive(x_new, df_X, LB, UB, eps):
    """
    Normalized distance used for proximity checks.
    Uses eps from CONFIG to avoid division by zero and
    as the closeness threshold.
    """
    X = df_X.values if hasattr(df_X, "values") else np.asarray(df_X)
    if X.size == 0:
        return np.array([])
    denom = np.maximum(UB - LB, eps)
    Xn = (X - LB) / denom
    xn = (x_new - LB) / denom
    d = np.linalg.norm(Xn - xn, axis=1)
    return d


def is_feasible_row(g_row, feas_tol):
    """Feasibility check using feasibility tolerance."""
    arr = np.asarray(g_row, float)
    if arr.size == 0:
        return True
    if np.isnan(arr).any():
        return False
    return np.all(arr <= feas_tol)


def sum_pos_violation(g_row):
    """Sum of positive constraint violations (NaNs treated as large penalty)."""
    arr = np.asarray(g_row, float)
    if arr.size == 0:
        return 0.0
    arr = np.where(np.isnan(arr), 1e6, arr)
    return float(np.sum(np.maximum(0.0, arr)))


def constrained_dominates(Fa, Ga, Fb, Gb, feas_tol, eps_dom=0.0):
    """
    Return True if (Fa, Ga) constrained-dominates (Fb, Gb).

    Rules (Deb-style constrained dominance + ε-dominance):
      1) Feasible vs infeasible:
           - Any feasible point dominates any infeasible point.
      2) Both feasible:
           - Apply ε-dominance in F-space:
               Fa_i <= Fb_i - eps_dom for all i, and
               Fa_j <  Fb_j - eps_dom for some j.
      3) Both infeasible:
           - The one with smaller total positive violation dominates.
    """
    Fa = np.asarray(Fa, float)
    Fb = np.asarray(Fb, float)
    Ga = np.asarray(Ga, float)
    Gb = np.asarray(Gb, float)

    viol_a = sum_pos_violation(Ga)
    viol_b = sum_pos_violation(Gb)

    feas_a = viol_a <= feas_tol
    feas_b = viol_b <= feas_tol

    # Case 1: feasibility first
    if feas_a and not feas_b:
        return True
    if feas_b and not feas_a:
        return False

    # Case 2: both feasible → ε-dominance in objectives
    if feas_a and feas_b:
        leq = np.all(Fa <= Fb - eps_dom)
        strict = np.any(Fa < Fb - eps_dom)
        return bool(leq and strict)

    # Case 3: both infeasible → compare violation
    if viol_a < viol_b - feas_tol:
        return True
    return False


def constrained_nondominated_mask(F, G, feas_tol, eps_dom=0.0):
    """
    Constrained nondominated set under constrained_dominates.
    Returns boolean mask for rows of F/G that are nondominated.
    """
    F = np.asarray(F, float)
    if G is None:
        G = np.zeros((F.shape[0], 0))
    G = np.asarray(G, float)

    n = F.shape[0]
    dominated = np.zeros(n, dtype=bool)

    for i in range(n):
        if dominated[i]:
            continue
        for j in range(n):
            if i == j or dominated[i]:
                continue
            if constrained_dominates(F[j], G[j], F[i], G[i], feas_tol, eps_dom):
                dominated[i] = True
                break

    return ~dominated


def is_nondominated(F):
    """
    Standard (unconstrained) Pareto nondominance mask in objective space.
    F: (n_points, n_obj)
    """
    F = np.asarray(F, float)
    n = F.shape[0]
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        if dominated[i]:
            continue
        for j in range(n):
            if i == j or dominated[i]:
                continue
            if np.all(F[j] <= F[i]) and np.any(F[j] < F[i]):
                dominated[i] = True
                break
    return ~dominated


def euclidean_dist_matrix(A, B):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    diff = A[:, None, :] - B[None, :, :]
    return np.sqrt(np.sum(diff * diff, axis=2))


# ====================================================
# Handling analysis failure
# ====================================================

def safe_true_evaluate(eval_func, x, n_obj, n_con):
    """
    Robust wrapper around the 'true' evaluation function.

    Returns:
        F (1D, length n_obj),
        G (1D, length n_con, or length 0 if n_con==0),
        success (bool).
    On failure it returns NaNs in F/G (except unconstrained problems where G
    is zero-length) and success=False.
    """
    try:
        F_raw, G_raw = eval_func(x)
    except Exception as e:
        F = np.full((n_obj,), np.nan, dtype=float)
        G = np.full((n_con,), np.nan, dtype=float) if n_con > 0 else np.zeros((0,), dtype=float)
        print(f"[safe_true_evaluate] Exception during eval: {e}. Marking as failure.")
        return F, G, False

    F = np.atleast_1d(F_raw).astype(float).flatten()
    if F.size != n_obj:
        print(f"[safe_true_evaluate] F size mismatch ({F.size} != {n_obj}). Marking as failure.")
        F = np.full((n_obj,), np.nan, dtype=float)
        G = np.full((n_con,), np.nan, dtype=float) if n_con > 0 else np.zeros((0,), dtype=float)
        return F, G, False

    if n_con > 0:
        G = np.atleast_1d(G_raw).astype(float).flatten()
        if G.size != n_con:
            print(f"[safe_true_evaluate] G size mismatch ({G.size} != {n_con}). Marking as failure.")
            G = np.full((n_con,), np.nan, dtype=float)
            return F, G, False
    else:
        G = np.zeros((0,), dtype=float)

    finite_F = np.all(np.isfinite(F))
    finite_G = (G.size == 0) or np.all(np.isfinite(G))
    success = finite_F and finite_G

    if not success:
        print("[safe_true_evaluate] Non-finite F/G detected. Marking as failure.")
        if not finite_F:
            F[~np.isfinite(F)] = np.nan
        if G.size > 0 and not finite_G:
            G[~np.isfinite(G)] = np.nan

    return F, G, success


# ====================================================
# pydace model training (regressors + classifier)
# ====================================================

def _train_single_dace(X, y, label, min_points=5):
    """
    Train a single Dace Kriging model for one scalar output.

    - Filters to finite y.
    - Requires at least min_points samples after filtering.
    - Uses linear regression and Gaussian correlation (regr_linear + corr_gauss).
    """
    mask = np.isfinite(y)
    if mask.sum() < min_points:
        print(f"[train_surrogates] Skipping {label}: only {mask.sum()} finite points.")
        return None

    X_clean = X[mask]
    y_clean = y[mask]

    D = X_clean.shape[1]

    # Match your working example: theta, thetaL, thetaU
    theta0 = np.ones(D)
    thetaL = 1e-5 * np.ones(D)
    thetaU = 1e5 * np.ones(D)

    try:
        dace_obj = Dace(
            regr=regr_linear,
            corr=corr_gauss,
            theta=theta0,
            thetaL=thetaL,
            thetaU=thetaU,
        )
        dace_obj.fit(X_clean, y_clean)
        return dace_obj
    except Exception as e:
        print(f"[train_surrogates] Dace fit failed for {label}: {e}")
        return None


def train_surrogate_models(df_X, df_F, df_G):
    """
    Train pydace Kriging models for each objective and constraint.
    Optionally also trains a classifier 'clf_success' if any NaNs appear.

    Returns:
        models: dict mapping keys 'f1','f2',...,'g1','g2',...,'clf_success' -> Dace or None
    """
    X = df_X.values.astype(float)
    models = {}

    # Objectives
    for col in df_F.columns:
        y = df_F[col].values.astype(float)
        m = _train_single_dace(X, y, label=col)
        if m is not None:
            models[col] = m

    # Constraints
    for col in df_G.columns:
        y = df_G[col].values.astype(float)
        m = _train_single_dace(X, y, label=col)
        if m is not None:
            models[col] = m

    # Classifier: success vs analysis failure (NaNs in any F or G)
    any_nan_F = np.isnan(df_F.values).any()
    any_nan_G = np.isnan(df_G.values).any() if df_G.shape[1] > 0 else False
    if any_nan_F or any_nan_G:
        row_has_nan_F = np.any(np.isnan(df_F.values), axis=1)
        if df_G.shape[1] > 0:
            row_has_nan_G = np.any(np.isnan(df_G.values), axis=1)
        else:
            row_has_nan_G = np.zeros(df_F.shape[0], dtype=bool)
        row_has_nan = row_has_nan_F | row_has_nan_G
        y_clf = (~row_has_nan).astype(float)

        if np.unique(y_clf).size >= 2:
            print("[train_surrogates] Training classifier 'clf_success' for analysis success/failure (with Kriging).")
            m_clf = _train_single_dace(X, y_clf, label="clf_success")
            if m_clf is not None:
                models["clf_success"] = m_clf

    if len(models) == 0:
        print("[train_surrogates] No surrogate models trained.")
        return None

    return models


def _predict_all_surrogates(models, x, n_obj, n_con):
    """
    Evaluate all surrogate models at a single design point x.

    Returns:
        F_pred (n_obj,), G_pred (n_con,), clf_pred (float or None)
    """
    x2 = np.atleast_2d(x).astype(float)
    F_pred = np.full(n_obj, np.nan, dtype=float)
    G_pred = np.full(n_con, np.nan, dtype=float) if n_con > 0 else np.zeros((0,), dtype=float)

    # Objectives
    for j in range(n_obj):
        key = f"f{j+1}"
        m = models.get(key, None)
        if m is not None:
            try:
                y_hat, _ = m.predict(x2, return_mse=True)
                F_pred[j] = float(np.asarray(y_hat).flatten()[0])
            except Exception:
                F_pred[j] = np.nan

    # Constraints
    for j in range(n_con):
        key = f"g{j+1}"
        m = models.get(key, None)
        if m is not None:
            try:
                y_hat, _ = m.predict(x2, return_mse=True)
                G_pred[j] = float(np.asarray(y_hat).flatten()[0])
            except Exception:
                G_pred[j] = np.nan

    # Classifier
    clf_pred = None
    m_clf = models.get("clf_success", None)
    if m_clf is not None:
        try:
            y_hat, _ = m_clf.predict(x2, return_mse=True)
            clf_pred = float(np.asarray(y_hat).flatten()[0])
        except Exception:
            clf_pred = None

    return F_pred, G_pred, clf_pred


def _predict_all_surrogates_batch(models, X, n_obj, n_con):
    """
    Batched version of _predict_all_surrogates.

    X: (N, D)
    Returns:
        F_pred: (N, n_obj)
        G_pred: (N, n_con) or (N, 0)
        clf_pred: (N,) or None
    """
    X = np.atleast_2d(X).astype(float)
    k = X.shape[0]

    F_pred = np.full((k, n_obj), np.nan, dtype=float)
    G_pred = np.full((k, n_con), np.nan, dtype=float) if n_con > 0 else np.zeros((k, 0), dtype=float)

    # Objectives
    for j in range(n_obj):
        key = f"f{j+1}"
        m = models.get(key, None)
        if m is None:
            continue
        try:
            y_hat, _ = m.predict(X, return_mse=True)
            F_pred[:, j] = np.asarray(y_hat, dtype=float).reshape(-1)
        except Exception as e:
            print(f"[BatchSurrogate] Failed objective {key} prediction: {e}")

    # Constraints
    for j in range(n_con):
        key = f"g{j+1}"
        m = models.get(key, None)
        if m is None:
            continue
        try:
            y_hat, _ = m.predict(X, return_mse=True)
            G_pred[:, j] = np.asarray(y_hat, dtype=float).reshape(-1)
        except Exception as e:
            print(f"[BatchSurrogate] Failed constraint {key} prediction: {e}")

    # Classifier
    clf_pred = None
    m_clf = models.get("clf_success", None)
    if m_clf is not None:
        try:
            y_hat, _ = m_clf.predict(X, return_mse=True)
            clf_pred = np.asarray(y_hat, dtype=float).reshape(-1)
        except Exception as e:
            print(f"[BatchSurrogate] Failed classifier prediction: {e}")
            clf_pred = None

    return F_pred, G_pred, clf_pred


# ====================================================
# NSGA-II surrogate optimization wrappers (vectorized Problem)
# ====================================================

class _SingleObjectiveSurrogateProblem(Problem):
    """
    Vectorized single-objective problem on surrogate models.

    Modes:
      - kind="f", obj_idx = 0/1 for f1 or f2.
      - kind="tchebycheff": objective is Tchebycheff distance to a given z_target.
    """

    def __init__(self, LB, UB, models, n_obj, n_con,
                 kind="f", obj_idx=0,
                 z_star=None, nadir=None, z_target=None, weights=None):
        self.LB = np.asarray(LB, float)
        self.UB = np.asarray(UB, float)
        self.models = models

        # "True" number of objectives in the surrogate (e.g. 2 for f1,f2)
        self.n_obj_true = int(n_obj)
        # True number of constraints in the surrogate
        self.n_con = int(n_con)

        self.kind = kind
        self.obj_idx = obj_idx
        self.z_star = np.asarray(z_star, float) if z_star is not None else None
        self.nadir = np.asarray(nadir, float) if nadir is not None else None
        self.z_target = np.asarray(z_target, float) if z_target is not None else None
        self.weights = np.asarray(weights, float) if weights is not None else None

        n_var = len(self.LB)
        # inequality constraints = physical constraints + optional classifier
        n_constr = self.n_con + (1 if "clf_success" in (models or {}) else 0)

        super().__init__(
            n_var=n_var,
            n_obj=1,
            n_constr=n_constr,
            xl=self.LB,
            xu=self.UB
        )

    def _evaluate(self, X, out, *args, **kwargs):
        X = np.atleast_2d(X).astype(float)

        F_pred_all, G_pred_all, clf_pred_all = _predict_all_surrogates_batch(
            self.models, X, self.n_obj_true, self.n_con
        )

        # Objective
        if self.kind == "f":
            # obj_idx=0 -> f1, obj_idx=1 -> f2, etc.
            vals = F_pred_all[:, self.obj_idx]

        elif self.kind == "tchebycheff":
            # scaled F in [approx] [0,1] using z_star/nadir
            denom = np.maximum(self.nadir - self.z_star, 1e-12)
            F_scaled = (F_pred_all - self.z_star) / denom
            # Tchebycheff distance to z_target for each row
            vals = np.max(self.weights * np.abs(F_scaled - self.z_target), axis=1)

        else:
            raise ValueError(f"Unknown kind '{self.kind}'")

        out["F"] = vals.reshape(-1, 1)

        # Constraints: physical g <= 0
        g_list = []

        # Physical constraints
        if self.n_con > 0:
            g_list.append(G_pred_all)

        # Classifier-based constraint: 0.5 - clf_pred <= 0  ⇒ clf_pred >= 0.5
        if clf_pred_all is not None:
            g_list.append(0.5 - clf_pred_all.reshape(-1, 1))

        if len(g_list) > 0:
            out["G"] = np.hstack(g_list)


def _nsga2_single_obj(LB, UB, models, n_obj, n_con,
                      kind="f", obj_idx=0,
                      z_star=None, nadir=None, z_target=None, weights=None,
                      pop_size=60, n_gen=80, rng_seed=None):
    """
    Run NSGA-II on a single-objective surrogate problem and return the best x and its surrogate F,G.
    """
    problem = _SingleObjectiveSurrogateProblem(
        LB, UB, models, n_obj, n_con,
        kind=kind,
        obj_idx=obj_idx,
        z_star=z_star, nadir=nadir,
        z_target=z_target, weights=weights,
    )

    algorithm = NSGA2(pop_size=pop_size, eliminate_duplicates=True)
    res = minimize(
        problem,
        algorithm,
        ('n_gen', n_gen),
        seed=rng_seed,
        verbose=False,
    )

    # res.X might be a set of candidates; take best by F (already stored)
    X_res = np.atleast_2d(res.X)
    F_res = np.atleast_2d(res.F).reshape(-1)
    best_idx = int(np.nanargmin(F_res))
    x_best = X_res[best_idx]

    F_pred, G_pred, _ = _predict_all_surrogates(models, x_best, n_obj, n_con)
    return round_array(x_best), F_pred, G_pred


# ====================================================
# Distance-based subset selection (unchanged)
# ====================================================

def distance_based_subset_selection(df_Ax, df_AF, df_AG,
                                    df_cx, df_cF, df_cG,
                                    feas_tol, eps, eps_dom=0.0,
                                    z_star=None, nadir=None):
    """
    Select one candidate from df_cx based on constrained ε-dominance,
    feasibility, and distance in F-space (via normalized scaling).
    """
    X_archive = np.asarray(df_Ax.values, float) if len(df_Ax) > 0 else np.zeros((0, 0))
    F_archive = np.asarray(df_AF.values, float) if len(df_AF) > 0 else np.zeros((0, 0))
    G_archive = np.asarray(df_AG.values, float) if len(df_AG) > 0 else np.zeros((0, 0))

    X_cand = np.asarray(df_cx.values, float)
    F_cand = np.asarray(df_cF.values, float)
    G_cand = np.asarray(df_cG.values, float) if df_cG.shape[1] > 0 else np.zeros((len(df_cF), 0))

    m = X_cand.shape[0]
    if m == 0:
        raise ValueError("Candidate set empty")
    if m == 1:
        return X_cand[0].copy()

    n_archive = F_archive.shape[0]

    # --- Archive feasible mask ---
    if n_archive > 0:
        feas_mask_archive = (np.apply_along_axis(is_feasible_row, 1, G_archive, feas_tol)
                             if G_archive.size > 0 else np.ones(n_archive, dtype=bool))
    else:
        feas_mask_archive = np.array([], dtype=bool)

    # --- Build targets_F, targets_G from archive ---
    if n_archive > 0 and np.any(feas_mask_archive):
        F_feas_arch = F_archive[feas_mask_archive]
        F_feas_arch_q = round_to_precision(F_feas_arch)          # quantize
        G_feas_arch = G_archive[feas_mask_archive]
        nd_mask_arch = constrained_nondominated_mask(F_feas_arch_q, G_feas_arch, feas_tol, eps_dom)
        targets_F = F_feas_arch_q[nd_mask_arch]
        targets_G = G_feas_arch[nd_mask_arch]
    elif n_archive > 0:
        violations = (np.apply_along_axis(sum_pos_violation, 1, G_archive)
                      if G_archive.size > 0 else np.zeros(n_archive))
        best_idx = int(np.nanargmin(violations))
        f_best_q = round_to_precision(F_archive[[best_idx], :])  # quantize
        targets_F = f_best_q
        targets_G = G_archive[[best_idx], :] if G_archive.size > 0 else np.zeros((1, 0))
    else:
        F_cand_q = round_to_precision(F_cand)                    # quantize
        targets_F = np.zeros((0, F_cand_q.shape[1])) if F_cand_q.size > 0 else np.zeros((0, 0))
        targets_G = np.zeros((0, G_cand.shape[1]))

    # --- Candidate feasibility mask ---
    cand_feas_mask = (np.apply_along_axis(is_feasible_row, 1, G_cand, feas_tol)
                      if G_cand.size > 0 else np.ones(m, dtype=bool))

    if np.any(cand_feas_mask):
        F_feas_cand = F_cand[cand_feas_mask]
        F_feas_cand_q = round_to_precision(F_feas_cand)          # quantize
        G_feas_cand = G_cand[cand_feas_mask] if G_cand.size > 0 else np.zeros((F_feas_cand.shape[0], 0))
        nd_mask_cand = constrained_nondominated_mask(F_feas_cand_q, G_feas_cand, feas_tol, eps_dom)
        idxs_feas = np.where(cand_feas_mask)[0]
        front0_local = np.where(nd_mask_cand)[0]
        promising_idx = idxs_feas[front0_local].tolist()
    else:
        violations_c = (np.apply_along_axis(sum_pos_violation, 1, G_cand)
                        if G_cand.size > 0 else np.zeros(m))
        best_local = int(np.nanargmin(violations_c))
        promising_idx = [best_local]

    if len(promising_idx) == 1:
        return X_cand[promising_idx[0]].copy()

    X_prom = X_cand[promising_idx]
    F_prom = F_cand[promising_idx]
    F_prom_q = round_to_precision(F_prom)
    G_prom = G_cand[promising_idx] if G_cand.size > 0 else np.zeros((len(promising_idx), 0))

    # --- Normalization using fixed global ideal/nadir if provided,
    #     otherwise archive/candidate-based estimates.
    if z_star is not None and nadir is not None:
        ideal = np.asarray(z_star, dtype=float).reshape(-1)
        nadir_arr = np.asarray(nadir, dtype=float).reshape(-1)

        n_obj = F_prom_q.shape[1]
        if ideal.size != n_obj:
            if ideal.size > n_obj:
                ideal = ideal[:n_obj]
            else:
                ideal = np.pad(ideal, (0, n_obj - ideal.size), mode='constant')
        if nadir_arr.size != n_obj:
            if nadir_arr.size > n_obj:
                nadir_arr = nadir_arr[:n_obj]
            else:
                nadir_arr = np.pad(nadir_arr, (0, n_obj - nadir_arr.size), mode='constant')
        nadir = nadir_arr
    else:
        if n_archive > 0 and np.any(feas_mask_archive):
            F_arch_for_norm = round_to_precision(F_archive[feas_mask_archive])
            ideal = np.nanmin(F_arch_for_norm, axis=0)
            nadir = np.nanmax(F_arch_for_norm, axis=0)
        elif n_archive > 0:
            F_arch_for_norm = round_to_precision(F_archive)
            ideal = np.nanmin(F_arch_for_norm, axis=0)
            nadir = np.nanmax(F_arch_for_norm, axis=0)
        else:
            F_cand_for_norm = round_to_precision(F_cand)
            ideal = np.nanmin(F_cand_for_norm, axis=0)
            nadir = np.nanmax(F_cand_for_norm, axis=0)

    denom = np.maximum(nadir - ideal, eps)

    def normF(Farr):
        return (np.asarray(Farr, float) - ideal) / denom

    if targets_F.shape[0] == 0:
        # no archive targets → just pick the first promising candidate
        return X_prom[0].copy()

    targets_Fn = normF(targets_F)
    F_prom_n = normF(F_prom_q)

    # --- Constrained nondominance on union of targets & promising ---
    union_F = np.vstack([targets_F, F_prom_q])
    union_G = np.vstack([targets_G, G_prom])
    nd_union = constrained_nondominated_mask(union_F, union_G, feas_tol, eps_dom)

    n_targets = targets_F.shape[0]
    prom_union_indices = np.arange(n_targets, n_targets + F_prom.shape[0])

    # elites are those candidate positions that are nondominated in union
    elites_mask = nd_union[prom_union_indices]
    elites = np.where(elites_mask)[0]

    # --- robust fallback if elites is empty ---
    if elites.size == 0:
        elite_Fn = F_prom_n
        d = euclidean_dist_matrix(elite_Fn, targets_Fn)
        best_rel = int(np.argmax(np.min(d, axis=1)))
        return X_prom[best_rel].copy()

    if elites.size == 1:
        return X_prom[elites[0]].copy()

    elite_Fn = F_prom_n[elites]
    d = euclidean_dist_matrix(elite_Fn, targets_Fn)
    best_rel = int(np.argmax(np.min(d, axis=1)))
    return X_prom[elites[best_rel]].copy()


# ====================================================
# Helpers for G dataframes
# ====================================================

def pad_df_G(df_G, target_cols):
    """Pad constraint DF to target_cols with zeros, or create empty appropriately."""
    if df_G.shape[1] == target_cols:
        return df_G.copy()
    if target_cols == 0:
        return pd.DataFrame(np.zeros((len(df_G), 0)))
    if df_G.shape[1] == 0:
        return pd.DataFrame(
            np.zeros((len(df_G), target_cols)),
            columns=[f"g{i+1}" for i in range(target_cols)],
        )

    pad_w = target_cols - df_G.shape[1]
    zeros = np.zeros((len(df_G), pad_w))
    new = np.hstack([df_G.values, zeros])
    cols = [f"g{i+1}" for i in range(target_cols)]
    return pd.DataFrame(new, columns=cols)


def safe_append_G_row(df_G: pd.DataFrame, g_flat: np.ndarray) -> pd.DataFrame:
    """Safely append constraint vector to df_G, handling zero-column/unconstrained cases."""
    if not isinstance(g_flat, np.ndarray):
        g_flat = np.atleast_1d(g_flat)
    g_flat = g_flat.flatten()

    if df_G.shape[1] == 0:
        return pd.concat([df_G, pd.DataFrame(np.zeros((1, 0)))], ignore_index=True)

    k = df_G.shape[1]
    if g_flat.size == 0:
        row = [0.0] * k
    else:
        row = list(g_flat) + [0.0] * max(0, k - g_flat.size)
        if len(row) > k:
            row = row[:k]

    df_G.loc[len(df_G)] = row
    return df_G


# ====================================================
# Warm-up: per concept feasible-by-singleobj (NSGA-II + pydace)
# ====================================================

def find_feasible_by_singleobj(obj_idx, df_X, df_F, df_G, evaluate_function,
                               LB, UB, D,
                               feas_tol,
                               max_attempts=50, eps=1e-6,
                               rng_seed=None,
                               pop_size=60, n_gen=80):
    """
    Minimize objective f_{obj_idx} with surrogate models (pydace) using NSGA-II,
    until a feasible (or best effort) point is found or max_attempts reached.
    """
    attempt = 0
    while attempt < max_attempts:
        attempt += 1

        models = train_surrogate_models(df_X, df_F, df_G)
        if models is None:
            print(f"[Min f{obj_idx+1}] No surrogate models to build NSGA-II problem; aborting warm-up.")
            break

        n_obj = df_F.shape[1]
        n_con = df_G.shape[1]

        try:
            x_new, F_pred, G_pred = _nsga2_single_obj(
                LB, UB, models, n_obj, n_con,
                kind="f", obj_idx=obj_idx,
                pop_size=pop_size, n_gen=n_gen,
                rng_seed=rng_seed,
            )
        except Exception as e:
            print(f"[Min f{obj_idx+1}] NSGA-II crashed: {e}; retry.")
            continue

        # Avoid too-close-to-archive
        dists = normalized_distance_to_archive(x_new, df_X, LB, UB, eps)
        if dists.size > 0 and np.any(dists < eps):
            print(f"[Min f{obj_idx+1}] Too close to archive (attempt {attempt}); retry.")
            continue

        # Evaluate true model
        F_new, G_new, success = safe_true_evaluate(
            evaluate_function, x_new, n_obj, n_con
        )

        df_X.loc[len(df_X)] = round_array(x_new)
        df_F.loc[len(df_F)] = F_new.flatten()
        df_G = safe_append_G_row(df_G, np.atleast_1d(G_new).flatten())

        print(f"[Min f{obj_idx+1}] #{attempt}: X={round_array(x_new)}, "
              f"F_true={round_array(F_new.flatten())}, "
              f"G_true={round_array(np.atleast_1d(G_new).flatten())}")

        if success and (G_new.size == 0 or
                        np.all(np.atleast_1d(G_new).flatten() <= feas_tol)):
            print(f"[Min f{obj_idx+1}] feasible.")
            return x_new, F_new.flatten(), np.atleast_1d(G_new).flatten(), df_G

        print(f"[Min f{obj_idx+1}] infeasible or failed analysis; retrain.")

    print(f"[Min f{obj_idx+1}] max attempts reached without feasible.")
    return None


# ====================================================
# Tchebycheff: per-concept proposal (NSGA-II + pydace)
# ====================================================

def _build_z_targets(n_f, n_points):
    """Build a set of target zs along a line in objective space."""
    v_parallel = np.array([-1.0, 1.0]) / np.sqrt(2.0)
    s_vals = np.linspace(-1.0/np.sqrt(2.0), 1.0/np.sqrt(2.0), n_points)
    z_targets = []
    for s in s_vals:
        z = np.zeros(n_f)
        z[0:2] = s * v_parallel if n_f >= 2 else np.array([s])
        z_targets.append(z)
    return np.vstack(z_targets)


def _filter_candidates_far_from_archive(cand_X, df_X, LB, UB, eps):
    """Keep only candidates that are at least eps away from all archive points."""
    def is_far(x_row):
        if len(df_X) == 0:
            return True
        d = normalized_distance_to_archive(x_row, df_X, LB, UB, eps)
        return d.size == 0 or np.all(d >= eps)

    return np.array([x for x in cand_X if is_far(x)])


def _predict_symbolic_batch_surrogate(eligible_X, models, n_obj, n_con):
    """
    Batched surrogate predictions for candidate set.
    """
    Fp, Gp, _ = _predict_all_surrogates_batch(models, eligible_X, n_obj, n_con)
    return Fp, Gp


def _solve_one_z_target(zt, idx,
                        LB, UB, surrogate_models, n_obj, n_con,
                        z_star, nadir, w_default,
                        pop_size, n_gen, rng_seed):

    print(f"[TCH][Job {idx}] Solving z_target {zt}")

    try:
        x_best, _, _ = _nsga2_single_obj(
            LB, UB, surrogate_models, n_obj, n_con,
            kind="tchebycheff",
            obj_idx=0,  # unused in tchebycheff mode
            z_star=z_star,
            nadir=nadir,
            z_target=zt,
            weights=w_default,
            pop_size=pop_size,
            n_gen=n_gen,
            rng_seed=rng_seed,
        )
        return x_best

    except Exception as e:
        print(f"[TCH][Job {idx}] NSGA-II failed: {e}")
        return None

def tchebycheff_propose_one(df_X, LB, UB, D,
                            z_star, nadir, n_points,
                            df_AF_global, df_AG_global,
                            feas_tol,
                            eps, eps_dom,
                            surrogate_models,
                            rng_seed=None,
                            pop_size=60, n_gen=60):

    print("[TCH] >>> Starting tchebycheff_propose_one (pydace + NSGA-II)")

    if surrogate_models is None or len(surrogate_models) == 0:
        print("[TCH] No surrogate models available; returning None.")
        return None, None, None

    n_obj = df_AF_global.shape[1] if len(df_AF_global) > 0 else 2
    n_obj = int(n_obj)
    n_con = df_AG_global.shape[1] if df_AG_global.shape[1] > 0 else 0
    n_con = int(n_con)

    if n_obj == 0:
        n_obj = 2

    print("[TCH] Building z_targets...")
    z_targets = _build_z_targets(n_obj, n_points)
    print(f"[TCH] Created {len(z_targets)} z_targets.")

    w_default = np.ones(n_obj) / float(max(1, n_obj))

    print("[TCH] Starting NSGA-II runs in parallel...")

    # PARALLEL EXECUTION OF ALL NSGA-II Tchebycheff SOLVES
    results = Parallel(n_jobs=-1)(
        delayed(_solve_one_z_target)(
            zt, idx,
            LB, UB, surrogate_models, n_obj, n_con,
            z_star, nadir, w_default,
            pop_size, n_gen,
            rng_seed
        )
        for idx, zt in enumerate(z_targets)
    )

    # Filter valid candidates + remove duplicates
    cand_X = []
    for x_best in results:
        if x_best is not None and \
           not any(np.allclose(x_best, xx, atol=eps) for xx in cand_X):
            cand_X.append(x_best)
            print(f"[TCH] Candidate added: {round_array(x_best)}")

    print(f"[TCH] Total candidates collected: {len(cand_X)}")

    if len(cand_X) == 0:
        print("[TCH] No candidates found — returning None.")
        return None, None, None

    cand_X = np.vstack(cand_X)

    print("[TCH] Filtering candidates near archive...")
    eligible_X = _filter_candidates_far_from_archive(cand_X, df_X, LB, UB, eps)
    print(f"[TCH] Eligible candidates after filtering: {len(eligible_X)}")

    if eligible_X.size == 0:
        print("[TCH] All candidates too close to archive — returning None.")
        return None, None, None

    # Predict F and G using surrogate models
    F_pred, G_pred = _predict_symbolic_batch_surrogate(
        eligible_X, surrogate_models, n_obj, n_con
    )

    df_cx = pd.DataFrame(eligible_X)
    df_cF = pd.DataFrame(F_pred, columns=[f"f{i+1}" for i in range(F_pred.shape[1])])

    max_g_global = df_AG_global.shape[1]

    # Match dimensions of candidate G to global archive
    if max_g_global == 0:
        df_cG = pd.DataFrame(np.zeros((len(eligible_X), 0)))
    else:
        G_adj = G_pred
        if G_adj.shape[1] > max_g_global:
            G_adj = G_adj[:, :max_g_global]
        elif G_adj.shape[1] < max_g_global:
            pad = max_g_global - G_adj.shape[1]
            G_adj = np.hstack([G_adj, np.zeros((G_adj.shape[0], pad))])
        df_cG = pd.DataFrame(G_adj, columns=[f"g{i+1}" for i in range(max_g_global)])

    dummy_Ax = (pd.DataFrame(np.zeros((len(df_AF_global), 1)))
                if len(df_AF_global) > 0 else pd.DataFrame(np.zeros((0, 1))))

    chosen_row = distance_based_subset_selection(
        dummy_Ax, df_AF_global, df_AG_global,
        df_cx, df_cF, df_cG,
        feas_tol=feas_tol, eps=eps, eps_dom=eps_dom,
        z_star=z_star, nadir=nadir,
    )

    idx = None
    for i in range(len(eligible_X)):
        if np.allclose(chosen_row, eligible_X[i], atol=1e-12):
            idx = i
            break
    if idx is None:
        idx = 0

    return eligible_X[idx], F_pred[idx], (G_pred[idx] if G_pred.shape[1] > 0 else np.zeros((0,)))


def singleobj_propose_one(obj_idx, df_X,
                          LB, UB, D,
                          feas_tol,
                          eps,
                          surrogate_models,
                          max_attempts=5, rng_seed=None,
                          pop_size=60, n_gen=60):
    """
    Surrogate-based single-objective proposal (no true evaluation):
    - uses pre-trained pydace models (surrogate_models),
    - solves a NSGA-II single-objective problem to minimize f_{obj_idx},
    - returns (x, F_pred, G_pred) based on surrogate models.
    """
    if surrogate_models is None or len(surrogate_models) == 0:
        print(f"[SingleObj f{obj_idx+1}] No surrogate models; aborting.")
        return None, None, None

    n_obj = 2  # global assumption for f1,f2
    n_con = 0  # constraints handled per-archive for constrained problems

    attempt = 0
    while attempt < max_attempts:
        attempt += 1
        try:
            x_new, F_pred, G_pred = _nsga2_single_obj(
                LB, UB, surrogate_models, n_obj, n_con,
                kind="f",
                obj_idx=obj_idx,
                pop_size=pop_size,
                n_gen=n_gen,
                rng_seed=rng_seed,
            )
        except Exception as e:
            print(f"[SingleObj f{obj_idx+1}] NSGA-II crash: {e}; retry.")
            continue

        # Check closeness to archive
        dists = normalized_distance_to_archive(x_new, df_X, LB, UB, eps)
        if dists.size > 0 and np.any(dists < eps):
            print(f"[SingleObj f{obj_idx+1}] Too close to archive (attempt {attempt}); retry.")
            continue

        print(f"[SingleObj f{obj_idx+1}] Proposal x={round_array(x_new)}, "
              f"F_pred={round_array(F_pred)}, "
              f"G_pred={round_array(G_pred)}")
        return x_new, F_pred, G_pred

    print(f"[SingleObj f{obj_idx+1}] max attempts reached without proposal.")
    return None, None, None


# ====================================================
# Global ideal/nadir and Pareto plotting
# ====================================================

def global_ideal_nadir(archives_dict, feas_tol, eps_dom=0.0, alpha=0.0):
    """
    Compute global ideal and nadir across all concepts.
    """
    F_list, G_list = [], []

    # Collect all F and G across concepts
    for _, d in archives_dict.items():
        Fvals = np.asarray(d["df_F"].values, float)
        if d["df_G"].shape[1] > 0:
            Gvals = np.asarray(d["df_G"].values, float)
        else:
            Gvals = np.zeros((len(Fvals), 0), dtype=float)

        if Fvals.size == 0:
            continue

        F_list.append(Fvals)
        G_list.append(Gvals)

    if len(F_list) == 0:
        raise ValueError("global_ideal_nadir: archives_dict contains no F values.")

    F_all = np.vstack(F_list)
    F_all_q = round_to_precision(F_all)
    G_all = np.vstack(G_list)

    # Finite objectives based on quantized F
    finite_F = np.all(np.isfinite(F_all_q), axis=1)

    # Feasibility via feas_tol
    if G_all.shape[1] > 0:
        G_clean = np.where(np.isnan(G_all), np.inf, G_all)
        feas_G = np.all(G_clean <= feas_tol, axis=1)
    else:
        feas_G = np.ones(F_all.shape[0], dtype=bool)

    feas_mask = finite_F & feas_G
    n_feas = int(np.sum(feas_mask))

    if n_feas >= 2:
        F_feas_q = F_all_q[feas_mask]
        nd_mask_F = is_nondominated(F_feas_q)
        F_pareto_q = F_feas_q[nd_mask_F]
        ideal = np.min(F_pareto_q, axis=0)
        nadir = np.max(F_pareto_q, axis=0)
        ideal = round_to_precision(ideal)
        nadir = round_to_precision(nadir)
    elif n_feas == 1:
        f_q = F_all_q[feas_mask][0]
        ideal = 0.5 * f_q
        nadir = 1.5 * f_q
        ideal = round_to_precision(ideal)
        nadir = round_to_precision(nadir)
    else:
        if G_all.shape[1] > 0:
            G_violation = np.where(np.isnan(G_all), 1e6, G_all)
            violations = np.sum(np.maximum(G_violation, 0.0), axis=1)
            best_idx = int(np.nanargmin(violations))
            f_best_q = F_all_q[best_idx]
        else:
            F_for_score = np.where(np.isnan(F_all_q), np.inf, F_all_q)
            scores = np.sum(F_for_score, axis=1)
            best_idx = int(np.nanargmin(scores))
            f_best_q = F_all_q[best_idx]

        ideal = 0.5 * f_best_q
        nadir = 1.5 * f_best_q
        ideal = round_to_precision(ideal)
        nadir = round_to_precision(nadir)

    return ideal, nadir


def plot_and_save_global_pareto(outdir: Path, archives_dict,
                                feas_tol, eps_dom=0.0, show=True):
    Ffeas_list = []
    Gfeas_list = []
    for _, d in archives_dict.items():
        Fv = d["df_F"].values
        Gv = d["df_G"].values if d["df_G"].shape[1] > 0 \
            else np.zeros((len(d["df_F"]), 0))
        feas_G = (np.all(np.where(np.isnan(Gv), np.inf, Gv) <= feas_tol, axis=1)
                  if Gv.size > 0 else np.ones(len(Fv), bool))
        feas_F = np.all(np.isfinite(Fv), axis=1)
        mask = feas_G & feas_F
        Ffeas_list.append(Fv[mask])
        Gfeas_list.append(Gv[mask])

    if len(Ffeas_list) == 0:
        print("No feasible points to plot.")
        return

    Ffeas = np.vstack(Ffeas_list)
    Gfeas = np.vstack(Gfeas_list)
    nd = constrained_nondominated_mask(Ffeas, Gfeas, feas_tol, eps_dom)

    fig = plt.figure(figsize=(6, 6))
    plt.scatter(Ffeas[:, 0], Ffeas[:, 1],
                facecolors='none', edgecolors='green',
                s=60, label='Feasible')
    plt.scatter(Ffeas[nd, 0], Ffeas[nd, 1], s=60, label='Constrained ε-Pareto')
    plt.xlabel('f1')
    plt.ylabel('f2')
    plt.title('Constrained ε-Nondominated Solutions (Global)')
    plt.legend()
    plt.grid(True)
    outpath = outdir / "pareto_global.png"
    fig.savefig(outpath, dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)


# ====================================================
# Experiment orchestration
# ====================================================

def build_global_FG(archives):
    """Concatenate F/G across all concepts, padding G where needed."""
    allF, allG = [], []
    max_g = max(A["df_G"].shape[1] for A in archives.values())
    for _, A in archives.items():
        allF.append(A["df_F"])
        allG.append(pad_df_G(A["df_G"], max_g))
    return (
        pd.concat(allF, axis=0, ignore_index=True),
        pd.concat(allG, axis=0, ignore_index=True),
    )


def initialize_archives(problem, lhs_mult, n_obj):
    """
    Perform initial LHS sampling and true evaluation for each concept.
    Returns dict of archives keyed by concept name.
    """
    archives = {}
    for sec, cfg in problem["concepts"].items():
        D = int(cfg["D"])
        LB = np.array(cfg["xl"], float)
        UB = np.array(cfg["xu"], float)
        n0 = lhs_mult * D
        X0 = sample_lhs_qmc(n0, D, LB, UB)
        X0 = round_array(X0)

        # infer constraint dimension
        n_con = None
        for xi in X0:
            try:
                _, G_tmp = cfg["eval_func"](xi)
                G_tmp = np.atleast_2d(G_tmp)
                n_con = G_tmp.shape[1]
                break
            except Exception:
                continue
        if n_con is None:
            n_con = 0
            print(f"[Init] Warning: could not infer constraint dimension for {sec}; assuming unconstrained.")

        F0 = np.zeros((n0, n_obj))
        G0 = np.zeros((n0, n_con)) if n_con > 0 else np.zeros((n0, 0))

        for i in range(len(X0)):
            F_i, G_i, success = safe_true_evaluate(cfg["eval_func"], X0[i], n_obj, n_con)
            F0[i, :] = F_i
            if n_con > 0:
                G0[i, :] = G_i
            if not success:
                print(f"[Init] {sec}: LHS sample {i} analysis failed, storing NaNs in F/G.")

        df_X = pd.DataFrame(X0, columns=[f"x{i}" for i in range(D)])
        df_F = pd.DataFrame(F0, columns=[f"f{i+1}" for i in range(n_obj)])
        if n_con > 0:
            df_G = pd.DataFrame(G0, columns=[f"g{i+1}" for i in range(n_con)])
        else:
            df_G = pd.DataFrame(np.zeros((len(X0), 0)))

        archives[sec] = {
            "D": D,
            "LB": LB,
            "UB": UB,
            "df_X": df_X,
            "df_F": df_F,
            "df_G": df_G,
            "eval_func": cfg["eval_func"],
        }
        print(f"[Init] {sec}: {len(df_X)} LHS; constraints={df_G.shape[1]}")

    return archives


def warmup_archives(archives, feas_tol,
                    max_attempts_singleobj, eps, seed):
    """Per-concept warm-up: minimize f1 then f2 until feasible (NSGA-II + pydace)."""
    for sec, A in archives.items():
        print(f"\n[Warm-up] Concept: {sec} → minimize f1 until feasible")
        ret = find_feasible_by_singleobj(
            0, A["df_X"], A["df_F"], A["df_G"], A["eval_func"],
            A["LB"], A["UB"], A["D"],
            feas_tol=feas_tol,
            max_attempts=max_attempts_singleobj,
            eps=eps,
            rng_seed=seed,
        )
        if ret is not None:
            _, _, _, A["df_G"] = ret

        print(f"[Warm-up] Concept: {sec} → minimize f2 until feasible")
        ret = find_feasible_by_singleobj(
            1, A["df_X"], A["df_F"], A["df_G"], A["eval_func"],
            A["LB"], A["UB"], A["D"],
            feas_tol=feas_tol,
            max_attempts=max_attempts_singleobj,
            eps=eps,
            rng_seed=seed,
        )
        if ret is not None:
            _, _, _, A["df_G"] = ret


def apply_random_fallback(archives, no_max_samples,
                          feas_tol, eps, eps_dom, seed, alpha):
    """
    Random fallback when Tchebycheff returns no proposals.
    Returns updated (archives, df_AF_global, df_AG_global, z_star, nadir).
    """
    remaining = no_max_samples - sum(len(A["df_X"]) for A in archives.values())
    if remaining <= 0:
        print("No proposals from any concept and budget exhausted; stopping loop.")
        return archives, None, None, None, None

    print("No proposals from any concept; using random fallback sampling.")

    rand_sec = random.choice(list(archives.keys()))
    A_rand = archives[rand_sec]

    x_rand = sample_lhs_qmc(
        n_samples=1,
        dim=A_rand["D"],
        lb=A_rand["LB"],
        ub=A_rand["UB"],
    )[0]
    x_rand = round_array(x_rand)
    n_obj = A_rand["df_F"].shape[1]
    n_con = A_rand["df_G"].shape[1]
    F_rand, G_rand, success = safe_true_evaluate(
        A_rand["eval_func"], x_rand, n_obj, n_con
    )

    A_rand["df_X"].loc[len(A_rand["df_X"])] = round_array(x_rand)
    A_rand["df_F"].loc[len(A_rand["df_F"])] = F_rand.flatten()
    A_rand["df_G"] = safe_append_G_row(
        A_rand["df_G"],
        np.atleast_1d(G_rand).flatten(),
    )

    print(f"[Random Fallback] Concept={rand_sec}, "
          f"x={round_array(x_rand)}, "
          f"F_true={round_array(F_rand.flatten())}, "
          f"G_true={round_array(np.atleast_1d(G_rand).flatten())}, "
          f"success={success}")

    df_AF_global, df_AG_global = build_global_FG(archives)
    z_star, nadir = global_ideal_nadir(archives, feas_tol, eps_dom, alpha)
    print(f"[Global] z*={round_array(z_star)}, nadir={round_array(nadir)}")

    return archives, df_AF_global, df_AG_global, z_star, nadir


def select_and_evaluate_proposal(proposals, archives,
                                 df_AF_global, df_AG_global,
                                 feas_tol, eps, eps_dom, seed, alpha,
                                 z_star, nadir):
    """
    From per-concept proposals (in surrogate space),
    choose one globally using DSS and evaluate true model.
    """
    idx_to_concept = {i: p["concept"] for i, p in enumerate(proposals)}
    df_cx_union = pd.DataFrame(
        np.arange(len(proposals)).reshape(-1, 1), columns=["concept_idx"]
    )
    df_cF_union = pd.DataFrame(
        np.vstack([p["F_pred"] for p in proposals]),
        columns=["f1", "f2"],
    )

    max_g = df_AG_global.shape[1]
    G_union = []
    for p in proposals:
        g = np.atleast_1d(p["G_pred"]).astype(float)
        if max_g == 0:
            G_union.append(np.zeros((1, 0)))
        else:
            if g.shape[0] > max_g:
                g = g[:max_g]
            pad = max_g - g.shape[0]
            if pad > 0:
                g = np.hstack([g, np.zeros(pad)])
            G_union.append(g.reshape(1, -1))

    df_cG_union = pd.DataFrame(
        np.vstack(G_union),
        columns=[f"g{i+1}" for i in range(max_g)] if max_g > 0 else [],
    )

    dummy_Ax = pd.DataFrame(np.zeros((len(df_AF_global), 1))) \
        if len(df_AF_global) > 0 else pd.DataFrame(np.zeros((0, 1)))

    chosen_marker = distance_based_subset_selection(
        dummy_Ax, df_AF_global, df_AG_global,
        df_cx_union, df_cF_union, df_cG_union,
        feas_tol=feas_tol, eps=eps, eps_dom=eps_dom,
        z_star=z_star, nadir=nadir,
    )
    chosen_idx = int(np.round(chosen_marker[0])) \
        if chosen_marker.ndim == 1 else int(np.round(chosen_marker[0, 0]))
    chosen_sec = idx_to_concept.get(chosen_idx, proposals[0]["concept"])
    chosen = next(p for p in proposals if p["concept"] == chosen_sec)
    print(f"[Global DSS] Selected concept = {chosen_sec}")

    A = archives[chosen_sec]
    n_obj = A["df_F"].shape[1]
    n_con = A["df_G"].shape[1]
    F_true, G_true, success = safe_true_evaluate(
        A["eval_func"], chosen["x"], n_obj, n_con
    )

    A["df_X"].loc[len(A["df_X"])] = round_array(chosen["x"])
    A["df_F"].loc[len(A["df_F"])] = F_true.flatten()

    g_true = np.atleast_1d(G_true).flatten()
    if A["df_G"].shape[1] == 0 and n_con > 0:
        A["df_G"] = pd.DataFrame(
            np.zeros((len(A["df_F"]) - 1, n_con)),
            columns=[f"g{i+1}" for i in range(n_con)],
        )
    if A["df_G"].shape[1] < n_con:
        pad_cols = n_con - A["df_G"].shape[1]
        for k in range(pad_cols):
            A["df_G"][f"g{A['df_G'].shape[1] + k + 1}"] = 0.0

    A["df_G"] = safe_append_G_row(A["df_G"], g_true)

    print(f"[True] {chosen_sec}: success={success}, "
          f"F_true={round_array(F_true.flatten())}, "
          f"G_true={round_array(g_true)}")

    df_AF_global, df_AG_global = build_global_FG(archives)
    z_new, n_new = global_ideal_nadir(archives, feas_tol, eps_dom, alpha)
    print(f"[Global] z*={round_array(z_new)}, nadir={round_array(n_new)}")

    return archives, df_AF_global, df_AG_global, z_new, n_new


def archives_snapshot(archives):
    """Lightweight snapshot suitable for pickling."""
    snap = {}
    for sec, A in archives.items():
        snap[sec] = {
            "D": int(A["D"]),
            "LB": A["LB"].tolist(),
            "UB": A["UB"].tolist(),
            "df_X": A["df_X"],
            "df_F": A["df_F"],
            "df_G": A["df_G"],
        }
    return snap


def run_experiment(CONFIG):
    problem = CONFIG["problem"]

    # Problem name
    try:
        problem_key = [k for k, v in globals().items() if v is problem][0]
    except IndexError:
        problem_key = "unknown_problem"
    base_name = problem_key.replace("problem_", "")

    # Suffix based on #concepts
    try:
        num_concepts = len(problem["concepts"])
        suffix = "multi" if num_concepts > 1 else "single"
    except Exception:
        num_concepts = 1
        suffix = "single"

    run_folder_name = f"{base_name}_{suffix}"

    seeds = CONFIG["seeds"]
    eps = CONFIG["eps"]
    feas_tol = CONFIG["feas_tol"]
    eps_dom = CONFIG.get("eps_dom", 0.0)
    alpha = CONFIG.get("alpha", 0.0)
    max_attempts_singleobj = CONFIG["max_attempts_singleobj"]
    max_total_weights = CONFIG["max_total_weights"]
    results_root = Path(CONFIG["results_root"])
    plot_flag = CONFIG["plot"]

    concepts = problem["concepts"]
    D_per_concept = {sec: int(cfg["D"]) for sec, cfg in concepts.items()}
    D_total = sum(D_per_concept.values())

    lhs_per_dim = CONFIG.get("lhs_per_dim", 10)
    budget_per_dim = CONFIG.get("budget_per_dim", 50)

    lhs_mult = lhs_per_dim
    default_no_max_samples = budget_per_dim * D_total
    no_max_samples = int(CONFIG.get("no_max_samples", default_no_max_samples))

    print(f"[Config] lhs_per_dim={lhs_per_dim} → n0 = lhs_per_dim·D per concept.")
    print(f"[Config] budget_per_dim={budget_per_dim}, D_total={D_total} → "
          f"no_max_samples={no_max_samples} total evals across all concepts.")

    for seed in seeds:
        np.random.seed(seed)
        random.seed(seed)
        print("\n" + "=" * 70)
        print(f"Multi-Objective Multi-Concept (seed={seed})")
        print("=" * 70)

        run_dir = results_root / run_folder_name / f"Seed-{seed}"
        run_dir.mkdir(parents=True, exist_ok=True)
        (run_dir / "config.json").write_text(json.dumps(CONFIG, indent=2, default=str))

        # 1) Initialization
        n_obj = 2
        archives = initialize_archives(problem, lhs_mult, n_obj)

        # 2) Warm-up (NSGA-II + pydace)
        warmup_archives(archives, feas_tol,
                        max_attempts_singleobj, eps, seed)

        # 3) Global ideal / nadir and global archive
        z_star, nadir = global_ideal_nadir(archives, feas_tol, eps_dom, alpha)
        print(f"\n[Global] Initial ideal: {round_array(z_star)}, "
              f"nadir: {round_array(nadir)}")

        df_AF_global, df_AG_global = build_global_FG(archives)

        current_total = sum(len(A["df_X"]) for A in archives.values())
        print(f"\n[Loop] Current archive size after init+warmup: {current_total}")
        print(f"[Loop] Total eval budget (no_max_samples): {no_max_samples}")

        if current_total >= no_max_samples:
            print("[Loop] Budget already exhausted by initial + warm-up evaluations; skipping main loop.")
        else:
            iter_count = 0
            while sum(len(A["df_X"]) for A in archives.values()) < no_max_samples:
                iter_count += 1
                current_total = sum(len(A["df_X"]) for A in archives.values())
                print(f"\n=== Global Iteration {iter_count} ===")
                print("Archive sizes:",
                      {sec: len(A["df_X"]) for sec, A in archives.items()})
                print(f"Total evals so far: {current_total}/{no_max_samples}")

                # Extra single-objective proposals only once every max_total_weights iterations
                do_extra_single_obj = (iter_count % max_total_weights == 0)
                if do_extra_single_obj:
                    print(f"[Loop] Extra single-objective proposals enabled "
                          f"(iter {iter_count}, multiple of max_total_weights={max_total_weights}).")
                else:
                    print(f"[Loop] Skipping extra single-objective proposals this iteration.")

                # 3a) Per-concept proposals:
                proposals = []
                for sec, A in archives.items():
                    print(f"\n[Loop] Concept: {sec} → build surrogates (pydace)")
                    models = train_surrogate_models(A["df_X"], A["df_F"], A["df_G"])
                    if models is None:
                        print(f"[{sec}] Skipping proposals: no surrogates available.")
                        continue

                    # (1) Tchebycheff proposals along weight line
                    x_cand, Fp, Gp = tchebycheff_propose_one(
                        A["df_X"],
                        A["LB"], A["UB"], A["D"],
                        z_star, nadir, n_points=max_total_weights,
                        df_AF_global=df_AF_global, df_AG_global=df_AG_global,
                        feas_tol=feas_tol,
                        eps=eps,
                        eps_dom=eps_dom,
                        surrogate_models=models,
                        rng_seed=seed,
                    )
                    if x_cand is not None:
                        proposals.append({
                            "concept": sec,
                            "x": x_cand,
                            "F_pred": Fp,
                            "G_pred": (Gp if Gp is not None else np.zeros((0,))),
                        })
                        print(f"[{sec}] (TCH) Proposed x={round_array(x_cand)}, "
                              f"F_pred={round_array(Fp)}, "
                              f"G_pred={round_array(Gp) if Gp is not None else '[]'}")
                    else:
                        print(f"[{sec}] No eligible Tchebycheff candidate.")

                    # (2) Additional single-objective proposals (min f1, min f2)
                    if do_extra_single_obj:
                        x_f1, Fp_f1, Gp_f1 = singleobj_propose_one(
                            0,
                            A["df_X"],
                            A["LB"], A["UB"], A["D"],
                            feas_tol=feas_tol,
                            eps=eps,
                            surrogate_models=models,
                            max_attempts=max_attempts_singleobj,
                            rng_seed=seed,
                        )
                        if x_f1 is not None:
                            proposals.append({
                                "concept": sec,
                                "x": x_f1,
                                "F_pred": Fp_f1,
                                "G_pred": (Gp_f1 if Gp_f1 is not None else np.zeros((0,))),
                            })
                            print(f"[{sec}] (SingleObj f1) Proposed x={round_array(x_f1)}, "
                                  f"F_pred={round_array(Fp_f1)}, "
                                  f"G_pred={round_array(Gp_f1) if Gp_f1 is not None else '[]'}")

                        x_f2, Fp_f2, Gp_f2 = singleobj_propose_one(
                            1,
                            A["df_X"],
                            A["LB"], A["UB"], A["D"],
                            feas_tol=feas_tol,
                            eps=eps,
                            surrogate_models=models,
                            max_attempts=max_attempts_singleobj,
                            rng_seed=seed,
                        )
                        if x_f2 is not None:
                            proposals.append({
                                "concept": sec,
                                "x": x_f2,
                                "F_pred": Fp_f2,
                                "G_pred": (Gp_f2 if Gp_f2 is not None else np.zeros((0,))),
                            })
                            print(f"[{sec}] (SingleObj f2) Proposed x={round_array(x_f2)}, "
                                  f"F_pred={round_array(Fp_f2)}, "
                                  f"G_pred={round_array(Gp_f2) if Gp_f2 is not None else '[]'}")

                # 3b) Handle no proposals => random fallback
                if len(proposals) == 0:
                    archives, df_AF_global, df_AG_global, z_star, nadir = apply_random_fallback(
                        archives, no_max_samples,
                        feas_tol, eps, eps_dom, seed, alpha,
                    )
                    if df_AF_global is None:
                        break
                    continue
                
                # 3c) Global DSS selection among proposals + true evaluation
                archives, df_AF_global, df_AG_global, z_star, nadir = select_and_evaluate_proposal(
                    proposals, archives, df_AF_global, df_AG_global,
                    feas_tol, eps, eps_dom, seed, alpha,
                    z_star, nadir,
                )

                current_total = sum(len(A["df_X"]) for A in archives.values())
                if current_total >= no_max_samples:
                    print(f"[Loop] Reached eval budget: {current_total}/{no_max_samples}. Stopping.")
                    break

        archives_to_save = archives_snapshot(archives)
        with open(run_dir / "archives.pkl", "wb") as f:
            pickle.dump(archives_to_save, f, protocol=pickle.HIGHEST_PROTOCOL)
        if plot_flag:
            plot_and_save_global_pareto(run_dir, archives, feas_tol, eps_dom, show=True)

        final_total = sum(len(A["df_X"]) for A in archives.values())
        print(f"\nRun complete. Final total evals = {final_total} (budget {no_max_samples}). "
              f"Data saved to {run_dir}")


# ====================================================
# Script entry point
# ====================================================

def run_one_problem_seed(args):
    """
    Worker for a single (problem, seed) combination, suitable for ProcessPoolExecutor.
    """
    problem_name, problem, base_config, seed = args

    CONFIG = copy.deepcopy(base_config)
    CONFIG["problem"] = problem
    CONFIG["seeds"] = [seed]  # run_experiment expects a list of seeds

    print(f"[Worker] Starting problem={problem_name}, seed={seed}")
    run_experiment(CONFIG)
    print(f"[Worker] Finished problem={problem_name}, seed={seed}")

        # Unconstrained
        #("dtlz2",       problem_dtlz2),
        #("zdt3",        problem_zdt3),
        # ("simulator",   problem_simulator),

        # Constrained
        #("bnh",         problem_bnh),
        #("tnk",         problem_tnk),
        #("welded_beam", problem_welded_beam),

if __name__ == "__main__":
    # We only run ONE problem: dtlz2

    problem_name_all = ["welded_beam"]#["dtlz2","zdt3", "bnh", "tnk", "welded_beam"]
    problem_all = [problem_welded_beam]#[problem_dtlz2, problem_zdt3, problem_bnh, problem_tnk, problem_welded_beam]

    # Seeds 1..21 (inclusive)
    seeds = list(range(1,2))

    # Base CONFIG shared across all seeds (problem & seeds set per worker)
    base_CONFIG = {
        "lhs_per_dim": 10,
        "budget_per_dim": 50,

        # eps: proximity threshold + denominator safeguard
        "eps": 1e-6,
        # feas_tol: feasibility tolerance and all g<=0 checks for true evals
        "feas_tol": 1e-6,
        # eps_dom / alpha: still accepted but not critical with fixed ideal/nadir
        "eps_dom": 0.0,
        "alpha": 0.0,

        "max_attempts_singleobj": 30,
        "max_total_weights": 31,

        "results_root": "Results_DACE",
        "plot": True,
    }
    for problem_name, problem in zip(problem_name_all, problem_all):
        # Build tasks: one per seed
        for seed in seeds:
            tasks = [(problem_name, problem, base_CONFIG, seed)]

            # For an M1: use (CPU count - 1) workers, but never less than 1
            cpu_count = os.cpu_count() or 4
            n_workers = max(1, min(len(tasks), cpu_count - 1))

            print(f"Launching {len(tasks)} runs for {problem_name} with seeds {seed}")
            print(f"Using {n_workers} parallel worker processes on this M1.")

            # Run all seeds in parallel across multiple processes
            # with ProcessPoolExecutor(max_workers=n_workers) as executor:
            #     list(executor.map(run_one_problem_seed, tasks))
            results = Parallel(n_jobs=n_workers)(delayed(run_one_problem_seed)(task)for task in tasks)

        print(f"All {problem_name} runs finished.")


Launching 1 runs for welded_beam with seeds 1
Using 1 parallel worker processes on this M1.
[Worker] Starting problem=welded_beam, seed=1
[Config] lhs_per_dim=10 → n0 = lhs_per_dim·D per concept.
[Config] budget_per_dim=50, D_total=4 → no_max_samples=200 total evals across all concepts.

Multi-Objective Multi-Concept (seed=1)
[Init] welded_beam: 40 LHS; constraints=4

[Warm-up] Concept: welded_beam → minimize f1 until feasible
[Min f1] #1: X=[0.12500923 7.92540057 0.86937136 1.47917805], F_true=[1.49328656 2.25859129], G_true=[ 4.66948700e+04  4.20816105e+05 -1.35416882e+00 -1.71699588e+05]
[Min f1] infeasible or failed analysis; retrain.
[Min f1] #2: X=[0.12507101 7.10745245 2.95937972 1.56553093], F_true=[4.82753901 0.05410163], G_true=[ 4.00755306e+04  6.75932253e+03 -1.44045992e+00 -6.67758478e+05]
[Min f1] infeasible or failed analysis; retrain.
[Min f1] #3: X=[0.14198821 3.51203625 6.69376408 0.40535256], F_true=[2.36421421 0.01805635], G_true=[ 3.14564007e+04 -2.25039473e+03 -2.

KeyboardInterrupt: 